In [1]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from tqdm import tqdm

In [15]:
## Configs and Hyperparameters
CSV_PATH = "../../data/splits.csv"
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 1e-4
NUM_CLASSES = 6
IMAGE_SIZE = 224 # Swin-T default square resolution

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [16]:
## Custom Dataset Definition
class TrashDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['filepath']
        label = row['class_idx']
        
        # Load image and convert to RGB
        image = Image.open(img_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.long)

In [17]:
## Dataset Transformation
# Standard ImageNet normalization values for pre-trained models
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])

train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    normalize
])

val_test_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    normalize
])

In [18]:
## Load the dataset
print(f"Loading splits from {CSV_PATH}...")
df_splits = pd.read_csv(CSV_PATH)

df_train = df_splits[df_splits['split'] == 'train']
df_val = df_splits[df_splits['split'] == 'val']
df_test = df_splits[df_splits['split'] == 'test']

train_dataset = TrashDataset(df_train, transform=train_transforms)
val_dataset = TrashDataset(df_val, transform=val_test_transforms)
test_dataset = TrashDataset(df_test, transform=val_test_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}, Test size: {len(test_dataset)}")

Loading splits from ../../data/splits.csv...
Train size: 10500, Val size: 2250, Test size: 2250


In [19]:
## Model Initialization
# Load pre-trained Swin-T
weights = models.Swin_T_Weights.DEFAULT
model = models.swin_t(weights=weights)

# Modify the head for 6 classes
in_features = model.head.in_features
model.head = nn.Linear(in_features, NUM_CLASSES)
model = model.to(device)

Downloading: "https://download.pytorch.org/models/swin_t-704ceda3.pth" to C:\Users\USER/.cache\torch\hub\checkpoints\swin_t-704ceda3.pth


100%|████████████████████████████████████████████████████████████████████████████████| 108M/108M [01:57<00:00, 964kB/s]


In [20]:
## Loss and Optimizer 
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)

In [ ]:
## Training Loop
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    
    # --- Training Phase ---
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    train_bar = tqdm(train_loader, desc="Training")
    for inputs, labels in train_bar:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct_train += torch.sum(preds == labels.data)
        total_train += labels.size(0)
        
        train_bar.set_postfix(loss=loss.item())
        
    epoch_train_loss = running_loss / total_train
    epoch_train_acc = correct_train.double() / total_train
    
    # --- Validation Phase ---
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        val_bar = tqdm(val_loader, desc="Validation")
        for inputs, labels in val_bar:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct_val += torch.sum(preds == labels.data)
            total_val += labels.size(0)
            
    epoch_val_loss = val_loss / total_val
    epoch_val_acc = correct_val.double() / total_val
    
    print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f}")
    print(f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")
    
    scheduler.step(epoch_val_loss)
    
    # Save the best model
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), "../../models/fc211045_swint/best_swin_t.pth")
        print(">> Saved new best model!")

In [ ]:
## Testing
print("\nLoading best model for testing...")
model.load_state_dict(torch.load("../../models/fc211045_swint/best_swin_t.pth"))
model.eval()

test_correct = 0
test_total = 0

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="Testing"):
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        test_correct += torch.sum(preds == labels.data)
        test_total += labels.size(0)

test_acc = test_correct.double() / test_total
print(f"\nFinal Test Accuracy: {test_acc:.4f}")